# CIC-IDS2018 — Class Distribution Analysis

## Objectives
- Overall traffic-class distribution
- BENIGN vs attack distribution
- Attack-class imbalance
- Rare attack classes
- Per-file class distribution
- Class presence across source files


In [ ]:
!pip -q install kagglehub

from pathlib import Path
import os
import shutil
import zipfile
import kagglehub

DATASET_SLUG = "solarmainframe/ids-intrusion-csv"
DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN")

if not KAGGLE_API_TOKEN:
    raise RuntimeError(
        "KAGGLE_API_TOKEN was not found. In Colab: open Secrets, "
        "add KAGGLE_API_TOKEN, paste your Kaggle API token, and "
        "enable Notebook access."
    )

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print("Kaggle authentication configured.")
print("Downloading CIC-IDS2018 dataset...")

download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Kaggle download location:", download_path)

csvs = list(download_path.rglob("*.csv"))

if csvs:
    for src in csvs:
        dst = DATA_DIR / src.name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
else:
    archives = list(download_path.rglob("*.zip"))
    if not archives:
        raise FileNotFoundError(
            f"No CSV files or ZIP archive found under {download_path}"
        )

    archive = archives[0]
    print("Extracting:", archive.name)
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(DATA_DIR)

nested_csvs = list(DATA_DIR.rglob("*.csv"))
for src in nested_csvs:
    if src.parent != DATA_DIR:
        dst = DATA_DIR / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

files = sorted(DATA_DIR.glob("*.csv"))

print(f"DATA_DIR: {DATA_DIR}")
print(f"CSV files found: {len(files)}")

if not files:
    raise FileNotFoundError(
        "Dataset download completed, but no CSV files were found."
    )

for f in files:
    print(f" - {f.name}")


Kaggle authentication configured.


100%|██████████| 1.60G/1.60G [00:17<00:00, 100MB/s] 

Extracting files...


Kaggle download location: /root/.cache/kagglehub/datasets/solarmainframe/ids-intrusion-csv/versions/1
DATA_DIR: /content/CIC-IDS2018
CSV files found: 10
 - 02-14-2018.csv
 - 02-15-2018.csv
 - 02-16-2018.csv
 - 02-20-2018.csv
 - 02-21-2018.csv
 - 02-22-2018.csv
 - 02-23-2018.csv
 - 02-28-2018.csv
 - 03-01-2018.csv
 - 03-02-2018.csv


In [ ]:

from pathlib import Path
import gc
import pandas as pd
import numpy as np

RESULTS_DIR = Path("/content/results/cicids2018/09_class_distribution")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 500_000
RANDOM_STATE = 42

print("Results:", RESULTS_DIR)

TARGET_COLUMN = "Label"

# Resolve the target column without assuming exact whitespace/casing
header = pd.read_csv(files[0], nrows=0)
matches = [c for c in header.columns if c.strip().lower() == TARGET_COLUMN.lower()]

if not matches:
    raise KeyError(f"Could not find target column '{TARGET_COLUMN}'.")

TARGET_COLUMN = matches[0]
print("Target column:", repr(TARGET_COLUMN))


Results: /content/results/cicids2018/09_class_distribution
Target column: 'Label'


## 1. Overall and Per-File Class Distribution

In [3]:

overall_counts = {}
per_file_rows = []

for file in files:
    print(f"Processing {file.name}...")

    file_counts = {}

    for chunk in pd.read_csv(
        file,
        usecols=[TARGET_COLUMN],
        chunksize=CHUNK_SIZE,
        low_memory=True
    ):
        counts = chunk[TARGET_COLUMN].value_counts(dropna=False)

        for label, count in counts.items():
            key = "<NA>" if pd.isna(label) else str(label).strip()
            file_counts[key] = file_counts.get(key, 0) + int(count)

    file_total = sum(file_counts.values())

    for label, count in file_counts.items():
        per_file_rows.append({
            "file_name": file.name,
            "label": label,
            "record_count": count,
            "percentage": count / file_total * 100
        })
        overall_counts[label] = overall_counts.get(label, 0) + count

    gc.collect()

overall_class_distribution = pd.DataFrame(
    [{"label": k, "record_count": v} for k, v in overall_counts.items()]
).sort_values("record_count", ascending=False)

total_records = int(overall_class_distribution["record_count"].sum())
overall_class_distribution["percentage"] = (
    overall_class_distribution["record_count"] / total_records * 100
)

per_file_class_distribution = pd.DataFrame(per_file_rows)

display(overall_class_distribution)


Processing 02-14-2018.csv...
Processing 02-15-2018.csv...
Processing 02-16-2018.csv...
Processing 02-20-2018.csv...
Processing 02-21-2018.csv...
Processing 02-22-2018.csv...
Processing 02-23-2018.csv...
Processing 02-28-2018.csv...
Processing 03-01-2018.csv...
Processing 03-02-2018.csv...


,label,record_count,percentage
2,Benign,13484708,83.069712
9,DDOS attack-HOIC,686012,4.226033
8,DDoS attacks-LOIC-HTTP,576191,3.549504
5,DoS attacks-Hulk,461912,2.845512
15,Bot,286191,1.763020
0,FTP-BruteForce,193360,1.191154
1,SSH-Bruteforce,187589,1.155603
14,Infilteration,161934,0.997560
6,DoS attacks-SlowHTTPTest,139890,0.861763
3,DoS attacks-GoldenEye,41508,0.255701


In [4]:

benign_mask = overall_class_distribution["label"].str.upper().eq("BENIGN")

benign_records = int(
    overall_class_distribution.loc[benign_mask, "record_count"].sum()
)
attack_records = total_records - benign_records

benign_attack_summary = pd.DataFrame([
    {
        "traffic_type": "BENIGN",
        "record_count": benign_records,
        "percentage": benign_records / total_records * 100
    },
    {
        "traffic_type": "ATTACK",
        "record_count": attack_records,
        "percentage": attack_records / total_records * 100
    }
])

attack_distribution = overall_class_distribution.loc[
    ~benign_mask
].copy()

attack_distribution["percentage_of_attack_traffic"] = (
    attack_distribution["record_count"] / attack_records * 100
)

display(benign_attack_summary)
display(attack_distribution)


,traffic_type,record_count,percentage
0,BENIGN,13484708,83.069712
1,ATTACK,2748294,16.930288


,label,record_count,percentage,percentage_of_attack_traffic
9,DDOS attack-HOIC,686012,4.226033,24.961376
8,DDoS attacks-LOIC-HTTP,576191,3.549504,20.965406
5,DoS attacks-Hulk,461912,2.845512,16.807227
15,Bot,286191,1.763020,10.413406
0,FTP-BruteForce,193360,1.191154,7.035637
1,SSH-Bruteforce,187589,1.155603,6.825653
14,Infilteration,161934,0.997560,5.892164
6,DoS attacks-SlowHTTPTest,139890,0.861763,5.090067
3,DoS attacks-GoldenEye,41508,0.255701,1.510319
4,DoS attacks-Slowloris,10990,0.067702,0.399884


## 2. Class Presence Across Source Files

In [5]:

class_presence_matrix = (
    per_file_class_distribution
    .assign(present=1)
    .pivot_table(
        index="label",
        columns="file_name",
        values="present",
        aggfunc="max",
        fill_value=0
    )
    .astype(int)
)

class_file_presence = (
    class_presence_matrix.sum(axis=1)
    .rename("files_present")
    .reset_index()
    .sort_values("files_present")
)

display(class_presence_matrix)
display(class_file_presence)


file_name,02-14-2018.csv,02-15-2018.csv,02-16-2018.csv,02-20-2018.csv,02-21-2018.csv,02-22-2018.csv,02-23-2018.csv,02-28-2018.csv,03-01-2018.csv,03-02-2018.csv
label,,,,,,,,,,
Benign,1,1,1,1,1,1,1,1,1,1
Bot,0,0,0,0,0,0,0,0,0,1
Brute Force -Web,0,0,0,0,0,1,1,0,0,0
Brute Force -XSS,0,0,0,0,0,1,1,0,0,0
DDOS attack-HOIC,0,0,0,0,1,0,0,0,0,0
DDOS attack-LOIC-UDP,0,0,0,0,1,0,0,0,0,0
DDoS attacks-LOIC-HTTP,0,0,0,1,0,0,0,0,0,0
DoS attacks-GoldenEye,0,1,0,0,0,0,0,0,0,0
DoS attacks-Hulk,0,0,1,0,0,0,0,0,0,0


,label,files_present
1,Bot,1
6,DDoS attacks-LOIC-HTTP,1
5,DDOS attack-LOIC-UDP,1
4,DDOS attack-HOIC,1
7,DoS attacks-GoldenEye,1
10,DoS attacks-Slowloris,1
9,DoS attacks-SlowHTTPTest,1
8,DoS attacks-Hulk,1
15,SSH-Bruteforce,1
11,FTP-BruteForce,1


## 3. Save Results

In [6]:

summary = pd.DataFrame([{
    "total_records": total_records,
    "benign_records": benign_records,
    "attack_records": attack_records,
    "benign_percentage": benign_records / total_records * 100,
    "attack_percentage": attack_records / total_records * 100,
    "number_of_classes": len(overall_class_distribution),
    "number_of_attack_classes": len(attack_distribution)
}])

overall_class_distribution.to_csv(
    RESULTS_DIR / "overall_class_distribution.csv", index=False
)
benign_attack_summary.to_csv(
    RESULTS_DIR / "benign_vs_attack_distribution.csv", index=False
)
attack_distribution.to_csv(
    RESULTS_DIR / "attack_class_distribution.csv", index=False
)
per_file_class_distribution.to_csv(
    RESULTS_DIR / "per_file_class_distribution.csv", index=False
)
class_presence_matrix.to_csv(
    RESULTS_DIR / "class_presence_matrix.csv"
)
class_file_presence.to_csv(
    RESULTS_DIR / "class_file_presence.csv", index=False
)
summary.to_csv(
    RESULTS_DIR / "class_imbalance_summary.csv", index=False
)

print("Generated artifacts:")
for f in sorted(RESULTS_DIR.glob("*")):
    print(" -", f.name)


Generated artifacts:
 - attack_class_distribution.csv
 - benign_vs_attack_distribution.csv
 - class_file_presence.csv
 - class_imbalance_summary.csv
 - class_presence_matrix.csv
 - overall_class_distribution.csv
 - per_file_class_distribution.csv


## Conclusion

The CIC-IDS2018 feature and data-quality analysis is complete.

The analysis confirms that the dataset contains several characteristics that must be considered before machine-learning use. The feature space contains numerical fields with missing values, infinite values, substantial variation in scale, skewed distributions, extreme observations, and redundant or low-information features. These characteristics indicate that the raw dataset should not be passed directly into an ML pipeline without preprocessing.

The analysis also demonstrates that many extreme values are not necessarily erroneous network observations. Consequently, extreme-value detection should be treated as a diagnostic step rather than an automatic row-removal criterion.

Overall, the results establish the following preprocessing requirements for CIC-IDS2018:

- Handle missing and infinite values explicitly.
- Remove constant and unsuitable features.
- Evaluate highly redundant and strongly correlated features.
- Apply appropriate transformations or scaling where required by the selected ML algorithms.
- Preserve legitimate network-flow extremes unless they are demonstrated to be invalid.
- Perform preprocessing using a reproducible pipeline to prevent train/test leakage.

No observations or feature values were modified during this exploratory analysis. The results will therefore serve as the basis for the subsequent preprocessing and feature-selection stage.

The CIC-IDS2018 exploratory analysis is now considered **closed**, and further modifications to the raw dataset will be deferred to the preprocessing stage.